In [83]:
import pandas as pd
import numpy as np
data = pd.read_csv('/Users/lorenzoallegrini/Python_ML/Machine_Learning/datasets/digit-recognizer/train.csv')
data = np.array(data)

m, n = data.shape
data_dev = data[0:1000].T
Y_dev = data_dev[0]
X_dev = data_dev[1: n] / 255

data_train = data[1000:m].T
Y_train = data_train[0]
X_train = data_train[1:n] / 255

In [84]:
import numpy as np
import matplotlib.pyplot as plt

def init_params(hidden_size_l1, hidden_size_l2):
    W1 = np.random.randn(hidden_size_l1, 784) * np.sqrt(2 / 784)
    b1 = np.zeros((hidden_size_l1,1))
    W2 = np.random.randn(hidden_size_l2, hidden_size_l1) * np.sqrt(2 / hidden_size_l1)
    b2 = np.zeros((hidden_size_l2,1))
    W3 = np.random.randn(10, hidden_size_l2) * np.sqrt(2 / hidden_size_l2)
    b3 = np.zeros((10,1))
    return W1, b1, W2, b2, W3, b3


def ReLU(Z):
    return np.maximum(0,Z)


def softmax(Z):
    Z_stable = Z - np.max(Z, axis=0, keepdims=True)
    A = np.exp(Z_stable) / np.sum(np.exp(Z_stable), axis=0, keepdims=True)
    return A


def forward_prop(W1, b1, W2, b2, W3, b3, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = ReLU(Z2)
    Z3 = W3.dot(A2) + b3
    A3 = softmax(Z3)
    return Z1, A1, Z2, A2, Z3, A3


def one_hot(Y):
    Y = Y.astype(int).flatten()
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y


def ReLU_deriv(Z):
    return Z > 0


def back_prop(Z1, A1, Z2, A2, W1, W2, Z3, A3, W3, X, Y):
    m = X.shape[1]
    one_hot_Y = one_hot(Y)
    dZ3 = A3 - one_hot_Y
    dW3 = 1 / m * dZ3.dot(A2.T)
    db3 = 1 / m * np.sum(dZ3, axis = 1, keepdims = True)
    dZ2 = W3.T.dot(dZ3) * ReLU_deriv(Z2)
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2, axis = 1, keepdims = True)
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1, axis = 1, keepdims = True)
    return dW1, db1, dW2, db2, dW3, db3


def update_params(W1, b1, W2, b2, W3, b3, dW1, db1, dW2, db2, dW3, db3, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2 
    b2 = b2 - alpha * db2
    W3 = W3 - alpha * dW3
    b3 = b3 - alpha * db3
    return W1, b1, W2, b2, W3, b3


def get_predictions(A3):
    return np.argmax(A3, 0)


def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / Y.size


def gradient_descent(X, Y, alpha, iterations, hidden_size_l1, hidden_size_l2,  Continuing_training = False, save_path = None):
    print(f'Training with ({hidden_size_l1},{hidden_size_l2}) architecture and learning rate: {alpha}')
    if Continuing_training and save_path is not None:
            data = np.load(save_path)
            W1 = data['W1']
            b1 = data['b1']
            W2 = data['W2']
            b2 = data['b2']
            W3 = data['W3']
            b3 = data['b3'] 
            print('Resuming Training with saved weights')     
    else:
        W1, b1, W2, b2, W3, b3 = init_params(hidden_size_l1, hidden_size_l2)
        print('Training from scratch')
    for i in range(iterations):
        Z1, A1, Z2, A2, Z3, A3 = forward_prop(W1, b1, W2, b2, W3, b3, X)
        dW1, db1, dW2, db2, dW3, db3 = back_prop(Z1, A1, Z2, A2, W1, W2, Z3, A3, W3, X, Y)
        W1, b1, W2, b2, W3, b3 = update_params(W1, b1, W2, b2, W3, b3, dW1, db1, dW2, db2, dW3, db3, alpha)
        if i % 50 == 0:
            predictions = get_predictions(A3)
            print('Iteration: ', i)
            print(get_accuracy(predictions, Y))
    if save_path is not None:
        np.savez(save_path, W1=W1, b1=b1, W2=W2, b2=b2, W3=W3, b3=b3)
        print(f'Parameters saved to {save_path}')
    
    return W1, b1, W2, b2, W3, b3


def make_predictions(X, W1, b1, W2, b2, W3, b3):
    _, _, _, _, _, A3 = forward_prop(W1, b1, W2, b2, W3, b3, X)
    predictions = get_predictions(A3)
    return predictions


def test_prediction(index, W1, b1, W2, b2, W3, b3):
    current_image = X_train[:, index, None]
    prediction = make_predictions(X_train[:, index, None], W1, b1, W2, b2, W3, b3)
    label = Y_train[index]
    print("Prediction: ", prediction)
    print("Label: ", label)
    
    current_image = current_image.reshape((28, 28)) * 255
    plt.gray()
    plt.imshow(current_image, interpolation='nearest')
    plt.show()



In [85]:
architectures = [(32, 32), (64, 64), (128, 64), (128, 128)]
learning_rates = [0.01, 0.05, 0.1]

results = []

for h1, h2 in architectures:
    for learning_rate in learning_rates:
        W1, b1, W2, b2, W3, b3 = gradient_descent(
            X_train, Y_train,
            alpha=learning_rate,
            iterations=200,
            hidden_size_l1=h1,
            hidden_size_l2=h2
        )

        predictions = make_predictions(X_dev, W1, b1, W2, b2, W3, b3)
        accuracy = get_accuracy(predictions, Y_dev)

        results.append((accuracy, h1, h2, learning_rate))
        print('---------------')
        print(f"Hidden Layer Sizes: ({h1},{h2}), Learning_rate = {learning_rate}, Accuracy: {accuracy:.4f}")
        print('---------------')

sorted(results, reverse=True)

Training with (32,32) architecture and learning rate: 0.01
Training from scratch
Iteration:  0
0.08204878048780488
Iteration:  50
0.30241463414634145
Iteration:  100
0.47268292682926827
Iteration:  150
0.5694878048780487
---------------
Hidden Layer Sizes: (32,32), Learning_rate = 0.01, Accuracy: 0.6340
---------------
Training with (32,32) architecture and learning rate: 0.05
Training from scratch
Iteration:  0
0.06207317073170732
Iteration:  50
0.7480243902439024
Iteration:  100
0.8206097560975609
Iteration:  150
0.8567560975609756
---------------
Hidden Layer Sizes: (32,32), Learning_rate = 0.05, Accuracy: 0.8740
---------------
Training with (32,32) architecture and learning rate: 0.1
Training from scratch
Iteration:  0
0.10539024390243902
Iteration:  50
0.8359756097560975
Iteration:  100
0.883
Iteration:  150
0.898390243902439
---------------
Hidden Layer Sizes: (32,32), Learning_rate = 0.1, Accuracy: 0.9100
---------------
Training with (64,64) architecture and learning rate: 0.0

[(np.float64(0.915), 128, 128, 0.1),
 (np.float64(0.915), 64, 64, 0.1),
 (np.float64(0.912), 128, 64, 0.1),
 (np.float64(0.91), 32, 32, 0.1),
 (np.float64(0.898), 128, 128, 0.05),
 (np.float64(0.895), 128, 64, 0.05),
 (np.float64(0.889), 64, 64, 0.05),
 (np.float64(0.874), 32, 32, 0.05),
 (np.float64(0.803), 128, 128, 0.01),
 (np.float64(0.789), 128, 64, 0.01),
 (np.float64(0.747), 64, 64, 0.01),
 (np.float64(0.634), 32, 32, 0.01)]

In [86]:
results_df = pd.DataFrame(results, columns = ['Validation Accuracy', 'Hidden Layer 1', 'Hidden Layer 2', 'Learning Rate']).sort_values('Validation Accuracy', ascending = False)
display(results_df)
optimised_arch_lr = np.array(results_df.iloc[0, :])
optimal_h_l1, optimal_h_l2, optimal_lr = optimised_arch_lr[1:4]

,Validation Accuracy,Hidden Layer 1,Hidden Layer 2,Learning Rate
5,0.915,64,64,0.10
11,0.915,128,128,0.10
8,0.912,128,64,0.10
2,0.910,32,32,0.10
10,0.898,128,128,0.05
7,0.895,128,64,0.05
4,0.889,64,64,0.05
1,0.874,32,32,0.05
9,0.803,128,128,0.01
6,0.789,128,64,0.01


In [109]:
W1, b1, W2, b2, W3, b3 = gradient_descent(X_train, Y_train, optimal_lr, 5000, hidden_size_l1 = int(optimal_h_l1), hidden_size_l2 = int(optimal_h_l2), Continuing_training = True, save_path = 'mnist_optimal_params.npz')

Training with (64,64) architecture and learning rate: 0.1
Resuming Training with saved weights
Iteration:  0
0.9988536585365854
Iteration:  50
0.9989268292682927
Iteration:  100
0.9989268292682927
Iteration:  150
0.9989268292682927
Iteration:  200
0.9989512195121951
Iteration:  250
0.9991219512195122
Iteration:  300
0.9991219512195122
Iteration:  350
0.9991463414634146
Iteration:  400
0.9991707317073171
Iteration:  450
0.9992439024390244
Iteration:  500
0.9993170731707317
Iteration:  550
0.9993414634146341
Iteration:  600
0.9993414634146341
Iteration:  650
0.9993414634146341
Iteration:  700
0.9993414634146341
Iteration:  750
0.9993414634146341
Iteration:  800
0.9994146341463415
Iteration:  850
0.9994146341463415
Iteration:  900
0.9994146341463415
Iteration:  950
0.9994390243902439
Iteration:  1000
0.9994390243902439
Iteration:  1050
0.9994878048780488
Iteration:  1100
0.9994878048780488
Iteration:  1150
0.9995121951219512
Iteration:  1200
0.9995121951219512
Iteration:  1250
0.999560975

In [ ]:
# Saving weights
np.savez(
    "mnist_optimal_params.npz",
    W1=W1, b1=b1,
    W2=W2, b2=b2,
    W3=W3, b3=b3
)

In [110]:
predictions_dev = make_predictions(X_dev, W1, b1, W2, b2, W3, b3)
accuracy_dev = get_accuracy(predictions_dev, Y_dev)
print(f"Validation accuracy: {accuracy_dev}")


Validation accuracy: 0.981


In [111]:
misclassified = np.flatnonzero(predictions_dev != Y_dev)
print(f"Misclassified validation images: {len(misclassified)}")


Misclassified validation images: 19


In [113]:
test_data = pd.read_csv("datasets/digit-recognizer/test.csv")

X_kaggle_test = test_data.to_numpy(dtype=np.float32).T / 255.0

kaggle_predictions = make_predictions(
    X_kaggle_test, W1, b1, W2, b2, W3, b3
)

submission = pd.DataFrame({
    "ImageId": np.arange(1, len(kaggle_predictions) + 1),
    "Label": kaggle_predictions
})

submission.to_csv("submission.csv", index=False)
submission.head()

,ImageId,Label
0,1,2
1,2,0
2,3,9
3,4,9
4,5,3
